# Burgers' Equation — Physics-Informed Neural Network

Burgers' equation is one of the simplest nonlinear partial differential equations
that combines **nonlinear advection** with **diffusion**. Named after the Dutch
physicist Johannes Martinus Burgers (1895–1981), it serves as a toy model for the
Navier–Stokes equations and appears across fluid mechanics, gas dynamics, nonlinear
acoustics, and traffic flow.

## The Equation

### Viscous form

$$
\frac{\partial u}{\partial t} + u\,\frac{\partial u}{\partial x}
= \nu\,\frac{\partial^2 u}{\partial x^2}
$$

where:

- $u(x, t)$ — the field variable (e.g. velocity)
- $\nu > 0$ — the viscosity (diffusion) coefficient
- $u\,\partial_x u$ — the **nonlinear convection** term, which steepens the wave
- $\nu\,\partial_{xx} u$ — the **diffusion** term, which smooths the wave

### Inviscid form

Setting $\nu = 0$ gives:

$$
\frac{\partial u}{\partial t} + u\,\frac{\partial u}{\partial x} = 0
$$

This is the prototype scalar **conservation law**: even from perfectly smooth
initial data, the solution steepens and forms **shocks** (discontinuities) in
finite time.

## Why It Matters

- **Balance of effects.** It captures the competition between nonlinear
  steepening and diffusive smoothing in a single, tractable equation.
- **Model for Navier–Stokes.** It is essentially the momentum equation with the
  pressure term removed and reduced to one dimension.
- **Exactly solvable.** It is one of very few nonlinear PDEs with a closed-form
  solution.

## The Cole–Hopf Transformation

The viscous equation can be linearized into the **heat equation** via the
substitution:

$$
u = -2\nu\,\frac{\partial}{\partial x}\ln\phi = -2\nu\,\frac{\phi_x}{\phi}
$$

which yields:

$$
\frac{\partial \phi}{\partial t} = \nu\,\frac{\partial^2 \phi}{\partial x^2}
$$

Since the heat equation is linear and well understood, this transformation gives
exact solutions to Burgers' equation.

## Common Applications

- Simplified models in fluid dynamics and turbulence
- Gas dynamics and shock-wave formation
- Nonlinear acoustics
- Traffic-flow modeling
- A standard benchmark problem for numerical PDE methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
class BurgersNet(nn.Module):
    """Fully-connected network that approximates u(x, t) for Burgers' equation.

    Architecture: 2 → 20 → 30 → 30 → 20 → 20 → 1, Tanh activations throughout.
    Tanh is preferred over ReLU for PINNs because it is smooth (infinitely
    differentiable), which is required for computing higher-order derivatives
    via automatic differentiation.
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 20),  nn.Tanh(),
            nn.Linear(20, 30), nn.Tanh(),
            nn.Linear(30, 30), nn.Tanh(),
            nn.Linear(30, 20), nn.Tanh(),
            nn.Linear(20, 20), nn.Tanh(),
            nn.Linear(20, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class PINN:
    """Physics-Informed Neural Network solver for the viscous Burgers equation.

    PDE:  u_t + u·u_x = ν·u_xx,  x ∈ [-1, 1],  t ∈ [0, 1]
    BCs:  u(-1, t) = u(1, t) = 0
    IC:   u(x, 0) = -sin(πx)

    Training uses two sequential phases:
      1. Adam      — fast, stochastic pre-conditioning.
      2. L-BFGS    — second-order fine-tuning to convergence.
    """

    # Viscosity coefficient (standard benchmark: ν = 0.01/π)
    NU: float = 0.01 / torch.pi

    def __init__(self, h: float = 0.1, k: float = 0.1):
        """
        Args:
            h: Spatial grid spacing for collocation points.
            k: Temporal grid spacing for collocation points.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = BurgersNet().to(self.device)
        self.criterion = nn.MSELoss()
        self.iter = 0

        # --- Collocation grid (PDE residual is enforced at every node) ---
        x = torch.arange(-1.0, 1.0 + h, h)
        t = torch.arange(0.0,  1.0 + k, k)

        X_grid, T_grid = torch.meshgrid(x, t, indexing="ij")
        # Shape: (N_col, 2) — each row is a (x, t) point
        self.X_col = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1).to(self.device)
        self.X_col.requires_grad_(True)

        # --- Labeled training data: boundary + initial conditions ---
        self.X_train, self.y_train = self._build_training_data(x, t)
        self.X_train = self.X_train.to(self.device)
        self.y_train = self.y_train.to(self.device)

        # --- Optimizers ---
        # Adam handles the early, noisy optimization landscape well.
        self.optimizer_adam = torch.optim.Adam(self.model.parameters())

        # L-BFGS is a quasi-Newton method: very effective at driving residuals
        # to near machine precision, but needs a good initialization (hence Adam first).
        self.optimizer_lbfgs = torch.optim.LBFGS(
            self.model.parameters(),
            max_iter=50_000,
            max_eval=50_000,
            tolerance_grad=1e-7,
            tolerance_change=float(np.finfo(float).eps),
            line_search_fn="strong_wolfe",
            history_size=50,
        )

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build_training_data(
        self, x: torch.Tensor, t: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Assemble all labeled (X, y) pairs for the boundary and initial conditions."""

        # Left BC: u(-1, t) = 0
        X_l, T_l = torch.meshgrid(x[:1], t, indexing="ij")
        bc_left = torch.stack([X_l.ravel(), T_l.ravel()], dim=1)
        y_left  = torch.zeros(len(bc_left), 1)

        # Right BC: u(1, t) = 0
        X_r, T_r = torch.meshgrid(x[-1:], t, indexing="ij")
        bc_right = torch.stack([X_r.ravel(), T_r.ravel()], dim=1)
        y_right  = torch.zeros(len(bc_right), 1)

        # IC: u(x, 0) = -sin(πx)
        X_ic, T_ic = torch.meshgrid(x, t[:1], indexing="ij")
        ic   = torch.stack([X_ic.ravel(), T_ic.ravel()], dim=1)
        y_ic = (-torch.sin(torch.pi * ic[:, 0])).unsqueeze(1)

        return torch.cat([bc_left, bc_right, ic]), torch.cat([y_left, y_right, y_ic])

    def _loss(self) -> torch.Tensor:
        """Compute total loss = data loss (BCs + IC) + PDE residual loss.

        This method serves as the closure for both Adam and L-BFGS. It zeroes
        gradients, performs a forward pass, computes both loss components,
        runs backpropagation, and returns the scalar loss.
        """
        self.optimizer_adam.zero_grad()
        self.optimizer_lbfgs.zero_grad()

        # --- Data loss: enforce boundary and initial conditions ---
        y_pred    = self.model(self.X_train)
        loss_data = self.criterion(y_pred, self.y_train)

        # --- PDE residual loss ---
        u = self.model(self.X_col)  # (N_col, 1)

        # First derivatives: ∂u/∂x and ∂u/∂t via automatic differentiation.
        # grad_outputs=ones sums the Jacobian rows, giving per-point gradients
        # because each u_i depends only on its own input X_col[i].
        grad_1 = torch.autograd.grad(
            u, self.X_col,
            grad_outputs=torch.ones_like(u),
            create_graph=True,
            retain_graph=True,
        )[0]
        u_x = grad_1[:, 0]  # ∂u/∂x
        u_t = grad_1[:, 1]  # ∂u/∂t

        # Second spatial derivative: ∂²u/∂x²
        grad_2 = torch.autograd.grad(
            u_x, self.X_col,
            grad_outputs=torch.ones_like(u_x),
            create_graph=True,
            retain_graph=True,
        )[0]
        u_xx = grad_2[:, 0]  # ∂²u/∂x²

        # Residual of Burgers' equation: u_t + u·u_x − ν·u_xx = 0
        residual  = u_t + u.squeeze() * u_x - self.NU * u_xx
        loss_pde  = self.criterion(residual, torch.zeros_like(residual))

        loss = loss_data + loss_pde
        loss.backward()

        self.iter += 1
        if self.iter % 100 == 0:
            print(f"  iter {self.iter:>6d}  loss {loss.item():.6e}")

        return loss

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def train(self, adam_steps: int = 1000) -> None:
        """Run two-phase training: Adam pre-conditioning then L-BFGS fine-tuning."""
        self.model.train()

        print("Phase 1 — Adam optimizer")
        for _ in tqdm(range(adam_steps), desc="Adam"):
            self._loss()
            self.optimizer_adam.step()

        print("\nPhase 2 — L-BFGS optimizer")
        self.optimizer_lbfgs.step(self._loss)
        print("Done.")

    def predict(self, X: torch.Tensor) -> np.ndarray:
        """Evaluate the trained network on a set of (x, t) points.

        Args:
            X: Tensor of shape (N, 2) with columns [x, t].

        Returns:
            NumPy array of shape (N, 1) with predicted u values.
        """
        self.model.eval()
        with torch.no_grad():
            return self.model(X.to(self.device)).cpu().numpy()

In [ ]:
pinn = PINN(h=0.1, k=0.1)
pinn.train(adam_steps=1000)

In [ ]:
# --- Evaluate on a fine grid for visualization ---
nx, nt = 300, 150
x_plot = torch.linspace(-1.0, 1.0, nx)
t_plot = torch.linspace(0.0, 1.0, nt)

X_grid, T_grid = torch.meshgrid(x_plot, t_plot, indexing="ij")
X_eval = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1)

u_pred = pinn.predict(X_eval).reshape(nx, nt)

# --- Plot solution as a space-time heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap: u(x, t) over the full domain
im = axes[0].contourf(
    t_plot.numpy(), x_plot.numpy(), u_pred,
    levels=100, cmap="RdBu_r"
)
fig.colorbar(im, ax=axes[0], label="u(x, t)")
axes[0].set_xlabel("t")
axes[0].set_ylabel("x")
axes[0].set_title("PINN solution — Burgers' equation")

# Slice plot: u(x, t) at selected time snapshots
for t_snap in [0.0, 0.25, 0.5, 0.75, 1.0]:
    idx = int(t_snap * (nt - 1))
    axes[1].plot(x_plot.numpy(), u_pred[:, idx], label=f"t = {t_snap:.2f}")
axes[1].set_xlabel("x")
axes[1].set_ylabel("u(x, t)")
axes[1].set_title("Solution profiles at selected times")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()